# IndabaX South Sudan 2026 — Food Insecurity Forecasting: Starter Notebook

This notebook gets you loading the data and producing a valid submission file. Everything in between — exploring the data, preparing features, choosing and training a model — is up to you.

**Scoring:** this competition is scored on **AUC** (area under the ROC curve). Your model must output a *probability* that a county is at risk, not a hard 0/1 label.

## Step 1: Import Libraries

Add whatever else you end up needing (a different model, encoding approach, etc.) as you go.

In [27]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score

## Step 2: Load the Dataset

In [28]:
train = pd.read_csv('Train__4_.csv')
test = pd.read_csv('Test__1_.csv')
ss = pd.read_csv('SampleSubmission__11_.csv')
variables = pd.read_csv('VariableDefinitions__1_.csv')

print(f"Train dataset: {train.shape[0]} rows, {train.shape[1]} columns")
print(f"Test dataset: {test.shape[0]} rows, {test.shape[1]} columns")
train.head()

Train dataset: 3099 rows, 11 columns
Test dataset: 234 rows, 10 columns


,ID,state,county,population,start_year,start_month,prior_period_ipc_phase,prior_period_phase3plus_pct,prior_year_cereal_production_tonnes,prior_year_cereal_gap_tonnes,food_insecurity_risk
0,ID_XAJI0Y,Abyei,Abyei,69160.0,2023,12,Crisis,49.2,12642.311151,-4404.321583,1
1,ID_6DPBHS,Abyei,Abyei,70246.0,2024,4,Crisis,30.4,12642.311151,-4404.321583,1
2,ID_AHXTHV,Abyei,Abyei,70246.0,2024,9,Crisis,55.5,12642.311151,-4404.321583,1
3,ID_3A3ZMF,Central Equatoria,Juba,492966.0,2014,10,Stressed,15.0,19333.526316,-44684.973684,0
4,ID_8MDD4V,Central Equatoria,Juba,501659.0,2015,1,Stressed,10.1,19333.526316,-44684.973684,0


## Step 3: Explore the Dataset

Before doing anything else, understand what you're working with. At minimum, you should know:
- Are there any missing values? (Hint: there shouldn't be — read `VariableDefinitions.csv` and the problem statement to understand why.)
- What's the target balance? Is it even, or skewed toward one class?
- Read `VariableDefinitions.csv` carefully. Note which features are *lagged* (reflect the county's status the last time it was analysed, not the current period) — this matters for how you think about what the model is actually learning.

**TODO:** explore the dataset yourself here. Some things worth checking:
- `train.isnull().sum()`
- `train['food_insecurity_risk'].value_counts(normalize=True)`
- How the target balance varies by `state`
- Correlations or relationships between features

In [29]:
# TODO: explore the data
# Check target balance and missing values
print('Missing values:\n', train.isna().sum())
print('\nTarget distribution:\n', train['food_insecurity_risk'].value_counts(normalize=True).sort_index())
print('\nTarget by state:\n', train.groupby('state')['food_insecurity_risk'].mean().sort_values(ascending=False))

# Preview data structure
train.head()

Missing values:
 ID                                     0
state                                  0
county                                 0
population                             0
start_year                             0
start_month                            0
prior_period_ipc_phase                 0
prior_period_phase3plus_pct            0
prior_year_cereal_production_tonnes    0
prior_year_cereal_gap_tonnes           0
food_insecurity_risk                   0
dtype: int64

Target distribution:
 food_insecurity_risk
0    0.222652
1    0.777348
Name: proportion, dtype: float64

Target by state:
 state
Abyei                      1.000000
Unity                      0.950000
Northern Bahr El Ghazal    0.905000
Upper Nile                 0.879237
Jonglei                    0.865297
Warrap                     0.800000
Lakes                      0.790625
Eastern Equatoria          0.771875
Western Bahr El Ghazal     0.766667
Central Equatoria          0.681416
Western Equatoria          0.

,ID,state,county,population,start_year,start_month,prior_period_ipc_phase,prior_period_phase3plus_pct,prior_year_cereal_production_tonnes,prior_year_cereal_gap_tonnes,food_insecurity_risk
0,ID_XAJI0Y,Abyei,Abyei,69160.0,2023,12,Crisis,49.2,12642.311151,-4404.321583,1
1,ID_6DPBHS,Abyei,Abyei,70246.0,2024,4,Crisis,30.4,12642.311151,-4404.321583,1
2,ID_AHXTHV,Abyei,Abyei,70246.0,2024,9,Crisis,55.5,12642.311151,-4404.321583,1
3,ID_3A3ZMF,Central Equatoria,Juba,492966.0,2014,10,Stressed,15.0,19333.526316,-44684.973684,0
4,ID_8MDD4V,Central Equatoria,Juba,501659.0,2015,1,Stressed,10.1,19333.526316,-44684.973684,0


## Step 4: Prepare Your Features

`state`, `county`, and `prior_period_ipc_phase` are categorical — they'll need encoding before most models can use them (e.g., `LabelEncoder`, one-hot encoding, or whatever approach you choose).

**Remember:** your final model must make meaningful use of all the provided relevant features (see the problem statement) — not just the strongest one. If you drop a feature, be ready to justify why in your write-up.

**TODO:** encode your categorical features and assemble your final feature set for both `train` and `test`. Make sure you handle `test` consistently with however you fit your encoders on `train` (e.g., watch for categories in `test` that don't appear in `train`).

In [30]:
# TODO: prepare your features
SEED = 42
TARGET = 'food_insecurity_risk'
ID_COL = 'ID'

IPC_ORDER = ['Minimal', 'Stressed', 'Crisis', 'Emergency', 'Catastrophe']
ipc_map = {p: i for i, p in enumerate(IPC_ORDER)}

def engineer(df):
    df = df.copy()
    df['ipc_phase_ord'] = df['prior_period_ipc_phase'].map(ipc_map)
    df['month_sin'] = np.sin(2 * np.pi * df['start_month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['start_month'] / 12)
    df['log_population'] = np.log1p(df['population'])
    df['cereal_gap_per_capita'] = df['prior_year_cereal_gap_tonnes'] / df['population'].replace(0, np.nan)
    df['cereal_prod_per_capita'] = df['prior_year_cereal_production_tonnes'] / df['population'].replace(0, np.nan)
    df['cereal_self_sufficiency'] = df['prior_year_cereal_production_tonnes'] / (
        df['prior_year_cereal_production_tonnes'] - df['prior_year_cereal_gap_tonnes']
    ).replace(0, np.nan)
    return df

train_fe = engineer(train).sort_values(['start_year', 'start_month']).reset_index(drop=True)
test_fe = engineer(test)

# Historical county/state rates using prior values only
train_fe = train_fe.sort_values(['county', 'start_year', 'start_month'])
train_fe['county_hist_rate'] = (
    train_fe.groupby('county')[TARGET].apply(lambda s: s.shift().expanding().mean())
    .reset_index(level=0, drop=True)
)
train_fe = train_fe.sort_values(['start_year', 'start_month'])
train_fe['county_hist_rate'] = train_fe['county_hist_rate'].fillna(train_fe[TARGET].expanding().mean().shift())
train_fe['county_hist_rate'] = train_fe['county_hist_rate'].fillna(train_fe[TARGET].mean())

state_rate_map = train_fe.groupby('state')[TARGET].mean()
global_rate = train_fe[TARGET].mean()

test_fe['county_hist_rate'] = test_fe['county'].map(train_fe.groupby('county')[TARGET].mean())
test_fe['county_hist_rate'] = test_fe['county_hist_rate'].fillna(test_fe['state'].map(state_rate_map))
test_fe['county_hist_rate'] = test_fe['county_hist_rate'].fillna(global_rate)

train_fe['state_hist_rate'] = (
    train_fe.groupby('state')[TARGET].apply(lambda s: s.shift().expanding().mean())
    .reset_index(level=0, drop=True)
)
train_fe['state_hist_rate'] = train_fe['state_hist_rate'].fillna(train_fe[TARGET].expanding().mean().shift())
train_fe['state_hist_rate'] = train_fe['state_hist_rate'].fillna(train_fe[TARGET].mean())
test_fe['state_hist_rate'] = test_fe['state'].map(state_rate_map).fillna(global_rate)

# Additional engineered signals for real-world risk patterns
train_fe['risk_gap'] = train_fe['county_hist_rate'] - train_fe['state_hist_rate']
test_fe['risk_gap'] = test_fe['county_hist_rate'] - test_fe['state_hist_rate']

train_fe['ipc_gap_interaction'] = train_fe['ipc_phase_ord'] * train_fe['prior_period_phase3plus_pct']
test_fe['ipc_gap_interaction'] = test_fe['ipc_phase_ord'] * test_fe['prior_period_phase3plus_pct']

NUMERIC_FEATURES = [
    'log_population', 'start_year', 'month_sin', 'month_cos',
    'ipc_phase_ord', 'prior_period_phase3plus_pct',
    'prior_year_cereal_production_tonnes', 'prior_year_cereal_gap_tonnes',
    'cereal_gap_per_capita', 'cereal_prod_per_capita', 'cereal_self_sufficiency',
    'county_hist_rate', 'state_hist_rate', 'risk_gap', 'ipc_gap_interaction',
]
CAT_FEATURES = ['state', 'county']

enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
train_cat = enc.fit_transform(train_fe[CAT_FEATURES])
test_cat = enc.transform(test_fe[CAT_FEATURES])

X = pd.concat([
    train_fe[NUMERIC_FEATURES].reset_index(drop=True),
    pd.DataFrame(train_cat, columns=CAT_FEATURES),
], axis=1).replace([np.inf, -np.inf], np.nan).fillna(0)
y = train_fe[TARGET].reset_index(drop=True)

X_test = pd.concat([
    test_fe[NUMERIC_FEATURES].reset_index(drop=True),
    pd.DataFrame(test_cat, columns=CAT_FEATURES),
], axis=1).replace([np.inf, -np.inf], np.nan).fillna(0)

print('Prepared features:', X.shape)
X.head()

Prepared features: (3099, 17)


,log_population,start_year,month_sin,month_cos,ipc_phase_ord,prior_period_phase3plus_pct,prior_year_cereal_production_tonnes,prior_year_cereal_gap_tonnes,cereal_gap_per_capita,cereal_prod_per_capita,cereal_self_sufficiency,county_hist_rate,state_hist_rate,risk_gap,ipc_gap_interaction,state,county
0,10.059208,2014,-0.866025,0.5,2,21.4,464.500000,-1946.684211,-0.083302,0.019877,0.192644,0.777348,0.777348,0.000000,42.8,6.0,0.0
1,12.044800,2014,-0.866025,0.5,2,45.2,6096.552632,-13686.868421,-0.080411,0.035818,0.308165,0.000000,0.000000,0.000000,90.4,3.0,2.0
2,11.576135,2014,-0.866025,0.5,1,13.1,9772.105263,-2646.105263,-0.024840,0.091736,0.786917,0.500000,0.500000,0.000000,13.1,5.0,3.0
3,13.178931,2014,-0.866025,0.5,1,14.9,35493.078947,-27439.131579,-0.051860,0.067082,0.563989,0.333333,0.000000,0.333333,14.9,5.0,4.0
4,12.495813,2014,-0.866025,0.5,1,10.1,26730.657895,-4246.684211,-0.015892,0.100034,0.862910,0.250000,0.000000,0.250000,10.1,5.0,5.0


## Step 5: Train a Model

Since scoring is AUC, your model needs to output a **probability**, not a hard label — most classifiers give you this via `.predict_proba()` rather than `.predict()`.

Note the class imbalance you should have found in Step 3 — a model that just predicts the majority class every time will look deceptively fine on plain accuracy but score poorly on AUC. Think about how you want to handle this (e.g., `class_weight='balanced'` is one option, not the only one).

**TODO:**
- Split your training data so you have a way to validate before submitting
- Choose and train a model
- Get validation probabilities and check your AUC

In [31]:
# TODO: train your model and check validation AUC
# Use a realistic temporal holdout: the most recent months are the validation set.
periods = train_fe[['start_year', 'start_month']].drop_duplicates().sort_values(['start_year', 'start_month'])
n_val_periods = max(3, int(len(periods) * 0.15))
val_periods = periods.tail(n_val_periods)
val_mask = train_fe.set_index(['start_year', 'start_month']).index.isin(
    val_periods.set_index(['start_year', 'start_month']).index
)
val_mask = pd.Series(val_mask, index=train_fe.index).reset_index(drop=True)

X_tr, X_val = X[~val_mask], X[val_mask]
y_tr, y_val = y[~val_mask], y[val_mask]

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_val_s = scaler.transform(X_val)

logreg = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=SEED)
logreg.fit(X_tr_s, y_tr)
val_pred_lr = logreg.predict_proba(X_val_s)[:, 1]

hgb = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.05,
    max_depth=4,
    l2_regularization=1.0,
    class_weight='balanced',
    random_state=SEED,
    early_stopping=True,
    validation_fraction=0.15,
)
hgb.fit(X_tr, y_tr)
val_pred_hgb = hgb.predict_proba(X_val)[:, 1]

val_pred_ens = 0.5 * val_pred_lr + 0.5 * val_pred_hgb

model_scores = {
    'LogisticRegression': roc_auc_score(y_val, val_pred_lr),
    'HistGradientBoosting': roc_auc_score(y_val, val_pred_hgb),
    'Ensemble': roc_auc_score(y_val, val_pred_ens),
}

print('Validation AUC by model:')
for model_name, score in model_scores.items():
    print(f'  {model_name}: {score:.4f}')

# Check whether performance differs by state on the validation period
state_auc = (
    pd.DataFrame({
        'state': train_fe.loc[val_mask, 'state'].reset_index(drop=True).values,
        'actual': y_val.to_numpy(),
        'prob': val_pred_ens,
    })
    .groupby('state')
    .apply(lambda df: roc_auc_score(df['actual'], df['prob']))
    .sort_values(ascending=False)
)
print('\nValidation AUC by state:')
print(state_auc)

# Feature importance check: permutation importance on the validation split
from sklearn.inspection import permutation_importance
perm = permutation_importance(hgb, X_val, y_val, n_repeats=10, random_state=SEED, scoring='roc_auc')
feature_importance = pd.Series(perm.importances_mean, index=X_val.columns).sort_values(ascending=False)
print('\nTop validation features by permutation importance:')
print(feature_importance.head(10))

model = {'logreg': logreg, 'hgb': hgb, 'ensemble': None}
model['ensemble'] = lambda p: 0.5 * logreg.predict_proba(scaler.transform(X_val))[:, 1] + 0.5 * hgb.predict_proba(X_val)[:, 1]

# Use the highest-scoring model for the final submission
best_model = max(model_scores, key=model_scores.get)
val_pred = {
    'LogisticRegression': val_pred_lr,
    'HistGradientBoosting': val_pred_hgb,
    'Ensemble': val_pred_ens,
}[best_model]


Validation AUC by model:
  LogisticRegression: 0.9800
  HistGradientBoosting: 0.9830
  Ensemble: 0.9842

Validation AUC by state:
state
Jonglei                    0.980769
Lakes                      0.974359
Western Equatoria          0.939103
Western Bahr El Ghazal     0.928571
Eastern Equatoria          0.921053
Abyei                           NaN
Central Equatoria               NaN
Northern Bahr El Ghazal         NaN
Unity                           NaN
Upper Nile                      NaN
Warrap                          NaN
dtype: float64


C:\Users\RSS_NEWC\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_ranking.py:469: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
C:\Users\RSS_NEWC\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_ranking.py:469: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
C:\Users\RSS_NEWC\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_ranking.py:469: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
C:\Users\RSS_NEWC\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-pac


Top validation features by permutation importance:
ipc_gap_interaction                    0.023445
month_sin                              0.011985
prior_period_phase3plus_pct            0.007414
cereal_self_sufficiency                0.006211
county_hist_rate                       0.005409
month_cos                              0.001907
cereal_gap_per_capita                  0.000891
prior_year_cereal_production_tonnes    0.000624
state                                  0.000561
prior_year_cereal_gap_tonnes           0.000463
dtype: float64


## Step 6: Pick a Decision Threshold (for your write-up)

AUC tells you how well your model *ranks* risk overall — it doesn't tell you how it performs at the specific cutoff a real humanitarian responder would actually use to decide "flag this county or not."

**Required for your write-up (see the problem statement):** pick a decision threshold and report the F1 score, precision, and recall your model achieves at that threshold, with reasoning for why you chose it.

**This is for your write-up only** — your actual Zindi submission below must always be raw probabilities, never rounded or thresholded.

In [32]:
# TODO: pick a threshold, compute F1/precision/recall at that threshold, and justify your choice in your write-up
best_thr, best_f1 = 0.5, -1
for thr in np.arange(0.1, 0.91, 0.02):
    pred = (val_pred >= thr).astype(int)
    score = f1_score(y_val, pred)
    if score > best_f1:
        best_f1 = score
        best_thr = thr

final_pred = (val_pred >= best_thr).astype(int)
print(f'Best threshold: {best_thr:.2f} | F1: {best_f1:.4f}')
print(f'Precision: {precision_score(y_val, final_pred):.4f}')
print(f'Recall: {recall_score(y_val, final_pred):.4f}')

Best threshold: 0.58 | F1: 0.9820
Precision: 0.9834
Recall: 0.9807


## Step 7: Predict on the Test Set and Create Your Submission

This part must follow the exact format Zindi expects — `ID` plus a probability for `food_insecurity_risk`, matching `SampleSubmission.csv`.

In [33]:
# Replace `model` and `test_feat` with whatever you named your trained model and prepared test features
scaler_full = StandardScaler()
X_full_s = scaler_full.fit_transform(X)
logreg_full = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=SEED)
logreg_full.fit(X_full_s, y)

test_pred_lr = logreg_full.predict_proba(scaler_full.transform(X_test))[:, 1]

hgb_full = HistGradientBoostingClassifier(
    max_iter=hgb.n_iter_,
    learning_rate=0.05,
    max_depth=4,
    l2_regularization=1.0,
    class_weight='balanced',
    random_state=SEED,
)
hgb_full.fit(X, y)
test_pred_hgb = hgb_full.predict_proba(X_test)[:, 1]

test_proba = 0.5 * test_pred_lr + 0.5 * test_pred_hgb

submission = ss.copy()
submission['food_insecurity_risk'] = test_proba
submission.to_csv('submission.csv', index=False)
submission.head()

,ID,food_insecurity_risk
0,ID_Z1OV5V,0.976083
1,ID_XA8MKU,0.983671
2,ID_1GPT49,0.996352
3,ID_PUSRZA,0.964502
4,ID_EQOXBE,0.965108


## Where to go from here

- Try more than one model type and compare validation AUC before picking your final one
- Look at what your model considers important — does it match your intuition about what should drive food insecurity risk?
- Explore whether performance differs by state
- Engineer your own features from the raw columns (e.g., trends, interactions)
- Don't just report your AUC — explain in your write-up *why* your model is trustworthy (or where it might fail) for a real humanitarian responder

Good luck!